# Decompile one TikTok into CreativeIR v0.1

This is a single-pass Gemini Flash baseline for the MP4 collected by `01_single_video_exploration.ipynb`. It sends the video and the repository schema contract in one multimodal JSON request, validates the response against the original Draft 2020-12 schema, and writes auditable artifacts beside the source media.

Set `GEMINI_API_KEY` (or configure Application Default Credentials) before running the call. `GEMINI_MODEL` overrides the default `gemini-flash-latest`. Do not run this notebook against media you are not entitled to process.

In [ ]:
%pip install -q -U google-genai jsonschema

import copy
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, display
from jsonschema import Draft202012Validator

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
video_id = os.environ.get("GEMINI_VIDEO_ID", "7106594312292453675")
default_source_dir = repo_root / "data" / "exploration" / video_id
source_dir = Path(os.environ.get("GEMINI_SOURCE_DIR", default_source_dir)).expanduser().resolve()
video_path = source_dir / "video.mp4"
metadata_path = source_dir / "metadata.json"
schema_path = repo_root / "schemas" / "creative_ir_v0_1.json"
raw_path = source_dir / "creative_ir.raw.json"
parsed_path = source_dir / "creative_ir.parsed.json"
usage_path = source_dir / "creative_ir.usage.json"
note_path = source_dir / "creative_ir.implementation.md"
for path in (video_path, metadata_path, schema_path):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
schema = json.loads(schema_path.read_text(encoding="utf-8"))
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
source_dir.mkdir(parents=True, exist_ok=True)
print(f"video={video_path}")
print(f"schema={schema_path}")

## Resolve the repository schema for Gemini

Gemini structured output accepts a JSON-Schema subset. The resolver expands local `$ref`s and preserves the repository’s object properties, required fields, enums, array items, and numeric bounds. Unsupported descriptive and Draft-only keywords are removed; the returned object is still validated against the unmodified repository schema below.

In [ ]:
def resolve_local_ref(root, node):
    if isinstance(node, dict) and set(node) == {"$ref"}:
        ref = node["$ref"]
        if not ref.startswith("#/$defs/"):
            raise ValueError(f"Unsupported schema reference: {ref}")
        target = root
        for part in ref[2:].split("/"):
            target = target[part.replace("~1", "/").replace("~0", "~")]
        return resolve_local_ref(root, copy.deepcopy(target))
    if isinstance(node, dict):
        return {key: resolve_local_ref(root, value) for key, value in node.items()}
    if isinstance(node, list):
        return [resolve_local_ref(root, value) for value in node]
    return node

# Keep only the structured-output subset accepted by Gemini; the original schema remains authoritative for validation.
GEMINI_KEYS = {"type", "properties", "required", "items", "enum", "minItems", "maxItems"}
def to_gemini_schema(node):
    node = resolve_local_ref(schema, node)
    if not isinstance(node, dict):
        return node
    if "const" in node:
        return {"type": "string", "enum": [node["const"]]}
    result = {}
    for key in GEMINI_KEYS:
        if key not in node:
            continue
        value = node[key]
        if key == "properties":
            result[key] = {name: to_gemini_schema(child) for name, child in value.items()}
        elif key == "items":
            result[key] = to_gemini_schema(value)
        else:
            result[key] = value
    return result

gemini_schema = to_gemini_schema(schema)
# Gemini rejects this contract's full 193-property nested response schema. JSON MIME mode
# keeps the request structured while the complete repository schema remains authoritative
# and is enforced immediately after generation.
Draft202012Validator.check_schema(schema)
print("Expanded schema properties:", list(gemini_schema["properties"]))

## One multimodal Gemini Flash call

The prompt requires ordered, timestamped shots; exact OCR/dialogue; visual, camera, editing and audio observations; narrative roles; cross-modal relationships; and reconstruction prompts. Unknown or uncertain values must remain explicit rather than guessed. Gemini JSON MIME output is followed by strict validation against the complete repository schema.

In [ ]:
from google import genai
from google.genai import types

model_name = os.environ.get("GEMINI_MODEL", "gemini-flash-latest")
prompt_version = "gemini-single-pass-creative-ir-v0.1"
metadata_context = json.dumps({key: metadata.get(key) for key in ("source_url", "video_id", "creator", "caption", "hashtags", "duration_seconds", "publication_date")}, ensure_ascii=False)
repository_schema = json.dumps(schema, ensure_ascii=False, separators=(",", ":"))
prompt = f"""You are a meticulous audiovisual decompiler. Return only a CreativeIR v0.1 JSON object matching the repository contract below.

Analyze the entire video from 0 seconds to its true end. Use exact timestamps in seconds and create ordered, non-overlapping shot ranges whose boundaries explain every hard cut. Record: hook, narrative beats and roles; every visible action and subject; camera framing, angle, movement and composition; every legible on-screen text segment with exact OCR, timing, placement and role; spoken dialogue with exact words and timing (or explicit absent/uncertain); music/original sound and sound effects; transitions and pacing; and visual-to-speech/text relationships. Keep observed facts separate from inferred interpretations. For each shot and globally provide detailed model-agnostic reconstruction prompts and continuity constraints.

Use this collector metadata only for source identity and measurable facts: {metadata_context}
The complete repository CreativeIR v0.1 schema is authoritative for every nested field; follow it exactly:
{repository_schema}
Do not silently invent names, speech, OCR, audio identity, timestamps, or events unavailable in the media. The caption is not on-screen OCR: never copy caption text into a shot's text segments unless those exact words are visibly rendered in the pixels. Treat platform/watermark text separately and include it only when legible. Use the schema's explicit absent, uncertain, unknown, or not_applicable values where appropriate. Do not emit empty objects or omit required nested properties. Ensure decompilation.model is exactly {model_name!r}, prompt_version is {prompt_version!r}, schema_version is "0.1", annotator_type is "automated", and created_at is an RFC3339 timestamp."""

client = genai.Client()
uploaded = client.files.upload(file=str(video_path))
response = client.models.generate_content(
    model=model_name,
    contents=[uploaded, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0,
    ),
)
if not response.text:
    raise ValueError("Gemini returned an empty response")
raw_path.write_text(response.text, encoding="utf-8")
parsed = json.loads(response.text)
# Keep raw model text untouched; make run metadata deterministic in the validated parsed artifact.
parsed["decompilation"] = {
    "model": model_name,
    "prompt_version": prompt_version,
    "schema_version": "0.1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "pipeline_version": "issue-3-gemini-single-pass-v0.1",
    "annotator_type": "automated",
}
parsed_path.write_text(json.dumps(parsed, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Saved raw response: {raw_path}")
print(f"Saved parsed response: {parsed_path}")

In [ ]:
# Validate the parsed object against the exact repository Draft 2020-12 schema.
Draft202012Validator(schema).validate(parsed)

def assert_temporal_integrity(ir):
    duration = ir["source"]["observed"]["duration_seconds"]
    shots = ir["observed"]["shots"]
    assert shots, "at least one shot is required"
    previous_end = 0.0
    for shot in shots:
        start = shot["time_range"]["start_seconds"]
        end = shot["time_range"]["end_seconds"]
        assert 0 <= start < end <= duration + 0.05, (shot["shot_id"], start, end, duration)
        assert start >= previous_end - 0.05, "shot ranges must be ordered"
        previous_end = end
    assert abs(shots[0]["time_range"]["start_seconds"]) <= 0.05
    assert abs(shots[-1]["time_range"]["end_seconds"] - duration) <= 0.05
    assert ir["generation"]["shot_order"] == [shot["shot_id"] for shot in shots]

assert_temporal_integrity(parsed)
usage = getattr(response, "usage_metadata", None)
usage_dict = {}
if usage is not None:
    usage_dict = {key: value for key, value in vars(usage).items() if value is not None}
usage_record = {
    "model": model_name,
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "video_id": video_id,
    "usage_metadata": usage_dict,
    "cost": "Gemini API pricing is model/account dependent; no cost was returned by the SDK. Consult the model pricing page using the recorded model and token counts.",
}
usage_path.write_text(json.dumps(usage_record, indent=2) + "\n", encoding="utf-8")
print("Draft 2020-12 validation and temporal/reference checks passed.")
print(json.dumps(usage_record, indent=2))

In [ ]:
# Manual inspection: source video and the resulting IR are visible together.
video_url = "data:video/mp4;base64," + __import__("base64").b64encode(video_path.read_bytes()).decode("ascii")
ir_html = json.dumps(parsed, ensure_ascii=False, indent=2).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
display(HTML(f"<div style='display:flex;gap:24px;align-items:flex-start'><video controls style='width:360px;max-height:640px'><source src='{video_url}' type='video/mp4'></video><pre style='white-space:pre-wrap;max-height:640px;overflow:auto;flex:1'>{ir_html}</pre></div>"))

## Implementation note and recommendation

The baseline should be judged against the source video and the deterministic checks above. The observed run has five ordered shots with cuts near 5, 7, 19.5 and 22 seconds. **Recommendation: multi-pass-needed.** The broad structure and challenge/reveal cards are useful, but unsupported OCR in shot 1, exact hard-cut timestamps, small-frog counting and audio/shot-boundary evidence warrant deterministic temporal sampling plus focused OCR/audio passes before production use. This recommendation is persisted to `creative_ir.implementation.md` by the next cell.

In [ ]:
note = f"""# Gemini CreativeIR implementation note

- Video: `{video_id}`
- Model: `{model_name}`
- Prompt: `{prompt_version}`
- Parsed output: `creative_ir.parsed.json`
- Raw model response: `creative_ir.raw.json`
- Usage and cost record: `creative_ir.usage.json`
- Validation: repository `schemas/creative_ir_v0_1.json` with Draft 2020-12 plus ordered temporal/reference checks.

## Evidence

The executed source run reports {parsed["source"]["observed"]["duration_seconds"]:.2f} seconds, five ordered shots, and hard cuts near {", ".join(f'{shot["time_range"]["end_seconds"]:.2f}' for shot in parsed["observed"]["shots"][:-1])} seconds. The broad structure, challenge card OCR, reveal count and visual continuity match manual frame inspection.
The single pass also produced unsupported shot-1 OCR; exact watermark/OCR timing and audio identity remain high-risk details.

## Recommendation

multi-pass-needed

Use this single-pass result as a baseline; add deterministic frame/time sampling, scene-boundary detection, and focused OCR/audio passes before relying on the output operationally.
"""
note_path.write_text(note, encoding="utf-8")
print(note_path)